# AIAA 3102 — Assignment 1: CIFAR-10 Image Classification

## Running environment

The submitted `.joblib` files were produced with the environment required by the
course, and this notebook is meant to be re-run in that same environment:

| Package | Version |
| --- | --- |
| Python | 3.14.6 |
| NumPy | 2.4.6 |
| SciPy | 1.18.0 |
| scikit-learn | 1.9.0 |
| joblib | 1.5.3 |

## Data paths

* `CIFAR10_Data/x_train.npy`, `CIFAR10_Data/y_train.npy` — 50,000 labelled training images
* `validation_set_to_students/images.npy`, `labels.npy` — 5,000 labelled validation images

## Final method and why it was chosen

**The problem with raw pixels.** An MLP on flattened pixels tops out near 51% clean
accuracy here, so everything depends on the feature representation.

**Features: a two-layer learned filter bank plus an oriented-filter bank.**

*Stage 1* learns 8 principal-component filters from 5x5 grayscale patches of the training
images. *Stage 2* learns, for each stage-1 response map, 8 further principal-component
filters from 5x5 patches of that map; each stage-2 response is rectified and
average-pooled over a 4x4 grid of 8x8-pixel blocks. That is a fixed-weight two-layer
convolutional network — the composition of two nonlinear stages is what a single filter
bank cannot express — and it costs nothing at prediction time beyond two
`scipy.ndimage.convolve` banks.

The two-layer bank (1,152 values) is concatenated with a steerable
derivative-of-Gaussian bank (8 orientations x 2 scales, plus two Laplacian-of-Gaussian
filters, each rectified and pooled on an 8x8 grid: 1,152 values) and with per-cell mean
colour (192 values), giving 2,496 features.

Measured on the validation set with seed 42 and everything else held fixed
(reduced screening budget, single model, horizontal-flip TTA):

| Feature representation | Validation accuracy |
| --- | --- |
| oriented-filter bank only (1,344 values) | 54.9% |
| 51-filter oriented bank + per-cell L2 normalisation (6,912 values) | 55.1% |
| **two-layer learned bank + oriented bank (2,496 values)** | **57.7%** |

Widening the oriented bank from 18 to 51 filters, adding per-cell normalisation and
adding sqrt compression all moved the result by less than a point: that bank is
saturated. The two-layer bank is what produced a real gain, which is why it is the core
of this solution.

**Classifier.** An MLP (1,024 hidden units, Adam, `alpha=1e-4`) on the principal
components of the standardised features.

**Ensemble: bagging plus test-time augmentation.** Each submitted model is a soft-voting
ensemble of `n_members` MLPs that all see the same augmented training set (clean,
flipped/translated and corrupted copies) but start from different random initialisations.
A shared `StandardScaler` + `PCA` is fitted once and reused by every member, so the
projection is stored only once in the file. Every member is then evaluated once per *view*: the identity, each of the eight shifts of
1, 2, 3 and 4 pixels in both axes, and the same 16 shifted views again after a horizontal
flip. That is 33 views per member and 99 voting members per model. Each view shares the
*same fitted* head object, so the whole view bank is test-time augmentation and adds no
model size at all - the view transform is merely a padded shift, built from module-level
NumPy callables (`np.pad` + `np.take`) so the model still unpickles with nothing but
numpy, joblib and scikit-learn. Views are cheap on disk and expensive at prediction time,
which is why the evaluation cell takes minutes rather than seconds; the accuracy they buy
is large, and it is the biggest single lever in this solution.

Screening runs with the same recipe at reduced scale:

| Configuration | Validation accuracy |
| --- | --- |
| one MLP, no flip TTA | 57.7% |
| 3 bagged members + flip TTA | 59.4% |
| 6 bagged members + flip TTA | 59.7% |
| 5 members, each on its own data mixture (previous recipe) | 56.4% |

Members specialised on a single data mixture were clearly worse than members trained on
the union of the mixtures, which is why the final recipe bags over the full augmented
set rather than splitting it. Beyond three members the gain was small (+0.3 for doubling
the count), so three is what the final recipe uses.

**Augmentation.** Corruptions follow the CIFAR-10-C benchmark definitions (the 15
classical types plus the four extra CIFAR-10-C types and geometric shifts), implemented
with NumPy/SciPy only. The hidden test set mixes original and corrupted images and its
corruptions may differ in type or strength from the validation set, so the training set
deliberately contains clean, geometrically shifted and corrupted copies. Augmentation
only affects the training data, so it is never part of the pickled model.

The corrupted half of the validation set is where the errors are, so the corrupted part
of the training mixture was tuned directly (single MLP head, horizontal-flip TTA,
reduced screening budget):

| Training mixture | Validation accuracy |
| --- | --- |
| clean + shifted + corrupted (1-3) + corrupted (1-5) | 57.7% |
| ... plus more corrupted copies | 58.6% |
| ... plus corruption at severity 3-5 only | 58.4% |
| **... plus copies carrying two corruptions at once (1-3 twice)** | **58.9%** |

Stacking two corruptions on the same image was the only change that helped beyond noise,
so the final mixture adds 45,000 such copies. Layering at severity 1-5 instead (58.4%)
and simply adding every component at once (57.3%) were both worse: past a point the
extra corrupted images stop looking like their class and start diluting the clean data.

**Seeds.** The pipeline, mixtures and hyper-parameters are fixed; the three submitted
models are trained from scratch with the random seeds 42, 2026 and 3102.

**Runtime.** Run All learns the filter banks, extracts features for every mixture and
trains one MLP per member per seed on a CPU - roughly 45-60 minutes per seed on a
20-core desktop. Nothing here uses a GPU.


## Measured result

Run All on this machine (20 logical cores, no GPU) took 3.3 hours and wrote three 46 MB
model files. Re-running the unmodified `Model_Evaluation.ipynb` against them gives:

| Model file | Validation accuracy |
| --- | --- |
| `model_seed42.joblib` | 0.6642 |
| `model_seed2026.joblib` | 0.6678 |
| `model_seed3102.joblib` | 0.6684 |
| **average** | **0.6668** |

The screening numbers quoted in the tables above were measured on a reduced training
budget; these are the numbers from the full 50,000-image run.


## 1. Environment check

In [1]:
import os
import time

import joblib
import numpy as np
import scipy
import sklearn

print("Python  ", os.sys.version.split()[0])
print("NumPy   ", np.__version__)
print("SciPy   ", scipy.__version__)
print("sklearn ", sklearn.__version__)
print("joblib  ", joblib.__version__)


Python   3.14.6
NumPy    2.4.6
SciPy    1.18.0
sklearn  1.9.0
joblib   1.5.3


## 2. Data paths and loading

`y_train.npy` is stored as a column vector `(50000, 1)`, so it is flattened to
`(50000,)` before training.

In [2]:
DATA_DIR = "CIFAR10_Data"
VAL_DIR = "validation_set_to_students"

X_train_images = np.load(os.path.join(DATA_DIR, "x_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy")).ravel()

X_val_images = np.load(os.path.join(VAL_DIR, "images.npy"))
y_val = np.load(os.path.join(VAL_DIR, "labels.npy"))

print("train images ", X_train_images.shape, X_train_images.dtype)
print("train labels ", y_train.shape, np.unique(y_train))
print("val   images ", X_val_images.shape, X_val_images.dtype)
print("val   labels ", y_val.shape, np.unique(y_val))


train images  (50000, 32, 32, 3) uint8
train labels  (50000,) [0 1 2 3 4 5 6 7 8 9]
val   images  (5000, 32, 32, 3) uint8
val   labels  (5000,) [0 1 2 3 4 5 6 7 8 9]


## 3. Feature extraction

Everything below is built from `scipy.ndimage.convolve` plus NumPy reshaping and
pooling, so the saved model contains no custom class and can be unpickled with
only NumPy and scikit-learn available.

In [3]:
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder, StandardScaler


def _ft(func, **kw):
    """FunctionTransformer wrapper; the kwargs are pickled together with the model.

    Only module-level callables (numpy / scipy / scikit-learn) may appear here.
    Model_Evaluation.ipynb imports nothing but numpy, joblib and scikit-learn, so a
    callable defined in this notebook would not survive unpickling.
    """
    return FunctionTransformer(func, kw_args=kw or None, validate=False)


def reshape(shape):
    return _ft(np.reshape, shape=shape)


# ------------------------------------------------------- oriented filter bank
def oriented_derivative_kernel(angle_deg, sigma, size=7):
    """Steerable derivative-of-Gaussian filter, shaped (1, size, size, 1)."""
    axis = np.arange(size, dtype=np.float32) - size // 2
    yy, xx = np.meshgrid(axis, axis, indexing="ij")
    gauss = np.exp(-(xx ** 2 + yy ** 2) / (2.0 * sigma ** 2))
    gx = -(xx / sigma ** 2) * gauss
    gy = -(yy / sigma ** 2) * gauss
    theta = np.deg2rad(angle_deg)
    kernel = np.cos(theta) * gx + np.sin(theta) * gy
    kernel -= kernel.mean()
    kernel /= np.abs(kernel).sum() + 1e-8
    return kernel.astype(np.float32)[None, :, :, None]


def laplacian_kernel(sigma, size=7):
    """Laplacian of Gaussian, shaped (1, size, size, 1)."""
    axis = np.arange(size, dtype=np.float32) - size // 2
    yy, xx = np.meshgrid(axis, axis, indexing="ij")
    r2 = xx ** 2 + yy ** 2
    kernel = (r2 - 2.0 * sigma ** 2) / sigma ** 4 * np.exp(-r2 / (2.0 * sigma ** 2))
    kernel -= kernel.mean()
    kernel /= np.abs(kernel).sum() + 1e-8
    return kernel.astype(np.float32)[None, :, :, None]


def filter_pool_branch(kernel, cells):
    """|filter response|, averaged over channels, mean-pooled over a cells x cells grid."""
    size = 32 // cells
    return Pipeline([
        ("convolve", _ft(ndimage.convolve, weights=kernel)),
        ("abs", _ft(np.abs)),
        ("channel_mean", _ft(np.mean, axis=3)),
        ("reshape", reshape((-1, cells, size, cells, size))),
        ("pool", _ft(np.mean, axis=(2, 4))),
        ("flatten", reshape((-1, cells * cells))),
    ])


def colour_branch(cells):
    """Mean colour of every cell, so the model keeps the coarse colour layout."""
    size = 32 // cells
    return Pipeline([
        ("reshape", reshape((-1, cells, size, cells, size, 3))),
        ("pool", _ft(np.mean, axis=(2, 4))),
        ("flatten", reshape((-1, cells * cells * 3))),
    ])


# --------------------------------------------------- two-layer learned filters
def sample_patches(maps, patch, n_patch, rng, chunk=20000):
    """(n_img, H, W) -> (n_patch, patch*patch) mean-removed, unit-variance patches."""
    windows = np.lib.stride_tricks.sliding_window_view(maps, (patch, patch), axis=(1, 2))
    n_img, ny, nx = windows.shape[:3]
    rows = rng.integers(0, n_img, n_patch)
    ys = rng.integers(0, ny, n_patch)
    xs = rng.integers(0, nx, n_patch)
    patches = np.zeros((n_patch, patch * patch), np.float32)
    for i in range(0, n_patch, chunk):
        sl = slice(i, min(i + chunk, n_patch))
        block = windows[rows[sl], ys[sl], xs[sl]].reshape(-1, patch * patch)
        block = block - block.mean(1, keepdims=True)
        var = block.var(1, keepdims=True)
        keep = var.ravel() > 1e-3
        block[keep] /= np.sqrt(var[keep])
        patches[sl] = block
    return patches


def pca_filters(patches, n_components):
    """Top principal directions of a patch sample, shaped (n, patch, patch)."""
    x = patches.astype(np.float64)
    x -= x.mean(0, keepdims=True)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    side = int(round(np.sqrt(patches.shape[1])))
    return vt[:n_components].reshape(n_components, side, side).astype(np.float32)


def learn_two_layer_filters(images, n_l1, n_l2, patch, n_patch, n_images, seed):
    """Stage-1 filters from the images; stage-2 filters from each stage-1 map."""
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(images), min(n_images, len(images)), replace=False))
    gray = images[idx].mean(3).astype(np.float32)
    stage1 = pca_filters(sample_patches(gray, patch, n_patch, rng), n_l1)
    stage2 = []
    for k in range(n_l1):
        response = ndimage.convolve(gray, stage1[k], mode="reflect", axes=(1, 2))
        np.clip(response, 0.0, None, out=response)
        stage2.append(pca_filters(sample_patches(response, patch, n_patch, rng), n_l2))
    return stage1, stage2


def learned_pool(cells):
    size = 32 // cells
    return Pipeline([
        ("reshape", reshape((-1, cells, size, cells, size))),
        ("pool", _ft(np.mean, axis=(2, 4))),
        ("flatten", reshape((-1, cells * cells))),
    ])


def learned_response(kernel, cells):
    """Convolve a single (N, 32, 32) map, rectify, block-pool."""
    return Pipeline([
        ("convolve", _ft(ndimage.convolve, weights=kernel, mode="reflect", axes=(1, 2))),
        ("relu", _ft(np.clip, a_min=0.0, a_max=None)),
        ("pool", learned_pool(cells)),
    ])


def learned_branches(filters, cells):
    """One branch per stage-1 map; the stage-2 bank of a map lives inside its own
    branch, so the extractor never materialises an (N, 32, 32, n_l1) array."""
    stage1, stage2 = filters
    branches = []
    for i, kernel in enumerate(stage1):
        head = [(f"l2_{j}", learned_response(k, cells)) for j, k in enumerate(stage2[i])]
        head.append(("l1", learned_pool(cells)))
        branches.append((
            f"map{i}",
            Pipeline([
                ("convolve", _ft(ndimage.convolve, weights=kernel, mode="reflect", axes=(1, 2))),
                ("relu", _ft(np.clip, a_min=0.0, a_max=None)),
                ("head", FeatureUnion(head, n_jobs=1)),
            ]),
        ))
    return branches


# ------------------------------------------------------------ feature block
def build_feature_block(filters, cells=8, learned_cells=4, angles=8, sigmas=(1.2, 2.2),
                        laplacians=(1.6, 3.0)):
    """Raw uint8 (N, 32, 32, 3) -> pooled feature vector. Stateless: never fitted."""
    oriented = []
    for sigma in sigmas:
        for angle in np.linspace(0, 180, angles, endpoint=False):
            oriented.append((
                f"o{int(sigma * 10)}_{int(angle)}",
                filter_pool_branch(oriented_derivative_kernel(angle, sigma), cells),
            ))
    for sigma in laplacians:
        oriented.append((f"l{int(sigma * 10)}", filter_pool_branch(laplacian_kernel(sigma), cells)))

    return Pipeline([
        ("features", FeatureUnion([
            ("oriented", Pipeline([
                ("as_float32", _ft(np.asarray, dtype=np.float32)),
                ("filters", FeatureUnion(oriented, n_jobs=1)),
            ])),
            ("learned", Pipeline([
                ("as_float32", _ft(np.asarray, dtype=np.float32)),
                ("gray", _ft(np.mean, axis=3)),
                ("maps", FeatureUnion(learned_branches(filters, learned_cells), n_jobs=1)),
            ])),
            ("colour", Pipeline([
                ("as_float32", _ft(np.asarray, dtype=np.float32)),
                ("cells", colour_branch(cells)),
            ])),
        ], n_jobs=1)),
    ])


## 4. Corruption and augmentation suite

CIFAR-10-C style corruptions, NumPy/SciPy only, severity 1-5. Used to build the
corrupted parts of the training set.

In [4]:
"""CIFAR-10-C style corruption suite implemented with numpy/scipy only.

Everything here is written for the final notebook: it must run with just
Python 3.14 + numpy + scipy + scikit-learn + joblib, and no custom object
ends up inside the saved model.

Images are (N, 32, 32, 3) uint8. Each corruption takes a float batch in
[0, 1] and a severity in 1..5 (CIFAR-10-C convention) and returns float in
[0, 1].
"""

import numpy as np
from scipy import ndimage


H, W = 32, 32


def _lin(sev, lo, hi):
    """Map severity 1..5 onto the interval [lo, hi]."""
    return lo + (hi - lo) * (np.asarray(sev, dtype=np.float32) - 1.0) / 4.0


# Parameter tables follow the CIFAR-10-C benchmark (Hendrycks & Dietterich) so
# that training-time augmentations have the same severity ladder as the
# corruptions the hidden test set is built from.
GAUSS_SIGMA = [0.08, 0.12, 0.18, 0.26, 0.38]
SHOT_C = [60.0, 25.0, 12.0, 5.0, 3.0]
IMPULSE_AMOUNT = [0.03, 0.06, 0.09, 0.17, 0.27]
DEFOCUS_RADIUS = [1, 2, 3, 4, 6]
GLASS = [(0.7, 1, 1), (0.9, 2, 2), (1.1, 2, 2), (1.3, 3, 3), (1.5, 4, 4)]
MOTION = [(10, 3), (15, 5), (15, 8), (15, 12), (20, 15)]
ZOOM_MAX = [0.11, 0.16, 0.19, 0.22, 0.25]
SNOW_AMOUNT = [0.1, 0.2, 0.3, 0.4, 0.5]
FROST_SCALE = [0.5, 0.75, 1.0, 1.25, 1.5]
FOG_COVERAGE = [0.2, 0.3, 0.4, 0.5, 0.6]
BRIGHTNESS_DELTA = [0.1, 0.2, 0.3, 0.4, 0.5]
CONTRAST_FACTOR = [0.4, 0.3, 0.2, 0.1, 0.05]
ELASTIC_ALPHA = [6.0, 9.0, 12.0, 15.0, 18.0]
PIXELATE_SCALE = [0.6, 0.5, 0.4, 0.3, 0.25]
JPEG_QUALITY = [25, 18, 15, 10, 7]


def _tab(table, sev):
    return table[int(np.asarray(sev).reshape(-1)[0]) - 1]


# ---------------------------------------------------------------- noise ----
def gaussian_noise(x, sev, rng):
    sigma = _tab(GAUSS_SIGMA, sev)
    return x + rng.normal(0.0, sigma, x.shape).astype(np.float32)


def shot_noise(x, sev, rng):
    counts = _tab(SHOT_C, sev)
    return rng.poisson(np.clip(x, 0, 1) * counts) / counts


def impulse_noise(x, sev, rng):
    amount = _tab(IMPULSE_AMOUNT, sev)
    mask = rng.random(x.shape) < amount
    salt = rng.random(x.shape) < 0.5
    out = x.copy()
    out[mask] = np.where(salt[mask], 1.0, 0.0)
    return out


# ----------------------------------------------------------------- blur ----
def _disk_kernel(radius):
    r = int(radius)
    yy, xx = np.mgrid[-r : r + 1, -r : r + 1]
    k = ((yy**2 + xx**2) <= r * r).astype(np.float32)
    return k / k.sum()


def defocus_blur(x, sev, rng):
    kernel = _disk_kernel(_tab(DEFOCUS_RADIUS, sev))[None, :, :, None]
    return ndimage.convolve(x, kernel, mode="reflect")


def _line_kernel(size, angle_deg):
    k = np.zeros((size, size), np.float32)
    k[size // 2, :] = 1.0
    k = ndimage.rotate(k, angle_deg, reshape=False, order=1)
    k[k < 1e-3] = 0.0
    total = k.sum()
    return k / total if total > 0 else k


def motion_blur(x, sev, rng):
    size, angle = _tab(MOTION, sev)
    angles = np.linspace(angle, angle + 180, 6, endpoint=False)
    offsets = rng.choice(len(angles) + 1, len(x))
    out = np.empty_like(x)
    for j in range(len(angles) + 1):
        m = offsets == j
        if not m.any():
            continue
        if j == 0:
            out[m] = x[m]
        else:
            out[m] = ndimage.convolve(x[m], _line_kernel(size, angles[j - 1])[None, :, :, None], mode="reflect")
    return out


def glass_blur(x, sev, rng):
    _, delta, iters = _tab(GLASS, sev)
    d = int(delta)
    out = x
    for _ in range(int(iters)):
        off_y = rng.integers(-d, d + 1, out.shape[:-1])
        off_x = rng.integers(-d, d + 1, out.shape[:-1])
        ys = np.clip(np.arange(H)[None, :, None] + off_y, 0, H - 1)
        xs = np.clip(np.arange(W)[None, None, :] + off_x, 0, W - 1)
        rows = np.arange(len(out))[:, None, None]
        out = out[rows, ys, xs]
    return ndimage.gaussian_filter(out, sigma=(0, 0.5, 0.5, 0), mode="reflect")


def zoom_blur(x, sev, rng):
    factors = 1.0 + np.linspace(0.02, _tab(ZOOM_MAX, sev), 6)
    acc = np.zeros_like(x)
    for f in factors:
        zoomed = ndimage.zoom(x, (1, f, f, 1), order=1, mode="reflect")
        y0 = max(0, (zoomed.shape[1] - H) // 2)
        x0 = max(0, (zoomed.shape[2] - W) // 2)
        if zoomed.shape[1] >= H:
            zoomed = zoomed[:, y0 : y0 + H, x0 : x0 + W]
        else:
            pad_y = H - zoomed.shape[1]
            pad_x = W - zoomed.shape[2]
            zoomed = np.pad(
                zoomed,
                ((0, 0), (pad_y // 2, pad_y - pad_y // 2), (pad_x // 2, pad_x - pad_x // 2), (0, 0)),
                mode="edge",
            )
        acc += zoomed
    return acc / len(factors)


# -------------------------------------------------------------- weather ----
def fog(x, sev, rng):
    strength = _tab(FOG_COVERAGE, sev) / 1.4
    small = rng.normal(0.6, 0.25, (len(x), 4, 4, 3)).astype(np.float32)
    mask = ndimage.zoom(small, (1, H / 4, W / 4, 1), order=1, mode="reflect")
    mask = np.clip(mask, 0, 1)
    fog_color = np.array([0.75, 0.78, 0.80], dtype=np.float32)
    alpha = (mask * strength).astype(np.float32)
    return x * (1 - alpha) + fog_color * alpha


def snow(x, sev, rng):
    dep = _tab(SNOW_AMOUNT, sev) / 1.6
    layer = np.zeros_like(x)
    for _ in range(3):
        flakes = rng.normal(0, 0.06 + 0.12 * dep, x.shape).astype(np.float32)
        flakes[flakes < 0.25] = 0
        layer += ndimage.gaussian_filter(flakes, sigma=(0, 0.6, 0.6, 0), mode="reflect")
    return x * (1 - dep) + (x + layer) * dep


def frost(x, sev, rng):
    pattern = rng.normal(0, 1, (len(x), 4, 4, 3)).astype(np.float32)
    pattern = ndimage.zoom(pattern, (1, H / 4, W / 4, 1), order=1, mode="reflect")
    pattern = ndimage.gaussian_filter(np.abs(pattern), sigma=(0, 1.2, 1.2, 0), mode="reflect")
    pattern = np.clip(pattern * _tab(FROST_SCALE, sev), 0, 1)
    return x * (1 - pattern * 0.6) + pattern * 0.6


# ------------------------------------------------------------- digital ----
def brightness(x, sev, rng):
    delta = _tab(BRIGHTNESS_DELTA, sev)
    sign = rng.choice([-1.0, 1.0])
    return x + sign * delta


def contrast(x, sev, rng):
    factor = _tab(CONTRAST_FACTOR, sev)
    mean = x.mean(axis=(1, 2, 3), keepdims=True)
    return (x - mean) * factor + mean


def pixelate(x, sev, rng):
    scale = _tab(PIXELATE_SCALE, sev)
    n = max(2, int(round(H * scale)))
    small = ndimage.zoom(x, (1, n / H, n / W, 1), order=1, mode="reflect")
    return ndimage.zoom(small, (1, H / n, W / n, 1), order=0, mode="reflect")


_JPEG_LUMA = np.array(
    [
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99],
    ],
    dtype=np.float32,
)


def jpeg_at_quality(x, quality):
    """JPEG round-trip at an explicit quality factor (no severity ladder)."""
    from scipy.fft import dctn, idctn

    quality = float(quality)
    scale = 5000.0 / quality if quality < 50 else 200.0 - 2.0 * quality
    table = _JPEG_LUMA * scale / 100.0
    table_chroma = table * 1.4

    y = x * 255.0 - 128.0
    r, g, b = y[..., 0], y[..., 1], y[..., 2]
    yc = 0.299 * r + 0.587 * g + 0.114 * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 128.0
    cr = 0.5 * r - 0.418688 * g - 0.081312 * b + 128.0
    stacked = np.stack([yc, cb - 128.0, cr - 128.0], -1)

    blocks = stacked.reshape(len(x), H // 8, 8, W // 8, 8, 3).transpose(0, 1, 3, 5, 2, 4)
    coef = dctn(blocks, axes=(-2, -1), norm="ortho")
    q = np.stack([table, table_chroma, table_chroma])  # (3, 8, 8)
    q = np.repeat(q[None], len(x) * (H // 8) * (W // 8), axis=0).reshape(coef.shape)
    coef = np.round(coef / q) * q
    blocks = idctn(coef, axes=(-2, -1), norm="ortho")
    stacked = blocks.transpose(0, 1, 4, 2, 5, 3).reshape(len(x), H, W, 3)

    yc, cb, cr = stacked[..., 0], stacked[..., 1] + 128.0, stacked[..., 2] + 128.0
    r = yc + 1.402 * (cr - 128.0)
    g = yc - 0.344136 * (cb - 128.0) - 0.714136 * (cr - 128.0)
    b = yc + 1.772 * (cb - 128.0)
    return (np.stack([r, g, b], -1) + 128.0) / 255.0


def jpeg_compression(x, sev, rng):
    return jpeg_at_quality(x, _tab(JPEG_QUALITY, sev))


def elastic(x, sev, rng):
    alpha = _tab(ELASTIC_ALPHA, sev)
    disp = rng.normal(0, 1, (len(x), 4, 4, 2)).astype(np.float32)
    disp = ndimage.zoom(disp, (1, H / 4, W / 4, 1), order=1, mode="reflect")
    disp = ndimage.gaussian_filter(disp, sigma=(0, 2.0, 2.0, 0), mode="reflect") * alpha
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    coords = np.stack(
        [
            np.clip(yy[None] + disp[..., 0], 0, H - 1.001),
            np.clip(xx[None] + disp[..., 1], 0, W - 1.001),
        ],
        axis=-1,
    )
    return _bilinear(x, coords)


def _bilinear(images, coords):
    y, x = coords[..., 0], coords[..., 1]
    y0, x0 = np.floor(y).astype(np.int32), np.floor(x).astype(np.int32)
    y1, x1 = np.minimum(y0 + 1, H - 1), np.minimum(x0 + 1, W - 1)
    wy, wx = (y - y0)[..., None], (x - x0)[..., None]
    rows = np.arange(len(images))[:, None, None]
    tl = images[rows, y0, x0]
    tr = images[rows, y0, x1]
    bl = images[rows, y1, x0]
    br = images[rows, y1, x1]
    top = tl * (1 - wx) + tr * wx
    bottom = bl * (1 - wx) + br * wx
    return top * (1 - wy) + bottom * wy


SPECKLE_C = [0.06, 0.1, 0.15, 0.2, 0.25]
GAUSSIAN_BLUR_SIGMA = [1.0, 1.4, 1.9, 2.6, 3.6]
SPATTER_AMOUNT = [0.02, 0.04, 0.07, 0.11, 0.16]
SATURATE_FACTOR = [1.15, 1.5, 2.0, 2.6, 3.2]
TRANSLATE_FRAC = [0.06, 0.1, 0.15, 0.2, 0.27]
ROTATE_DEG = [10.0, 15.0, 20.0, 25.0, 30.0]


def speckle_noise(x, sev, rng):
    c = _tab(SPECKLE_C, sev)
    return x + x * rng.normal(0, c, x.shape).astype(np.float32)


def gaussian_blur(x, sev, rng):
    sigma = _tab(GAUSSIAN_BLUR_SIGMA, sev)
    return ndimage.gaussian_filter(x, sigma=(0, sigma, sigma, 0), mode="reflect")


def spatter(x, sev, rng):
    """Dark mud-like blobs, the classic CIFAR-10-C spatter look."""
    amount = _tab(SPATTER_AMOUNT, sev)
    out = x.copy()
    for i in range(len(x)):
        n = rng.integers(3, 7)
        for _ in range(n):
            cy, cx = rng.integers(0, 32, 2)
            r = rng.integers(2, 6)
            yy, xx = np.mgrid[0:32, 0:32]
            m = ((yy - cy) ** 2 + (xx - cx) ** 2) <= r * r
            out[i][m] = out[i][m] * (1 - amount * 4) + np.array([0.25, 0.2, 0.15]) * amount * 4
    return out


def saturate(x, sev, rng):
    factor = _tab(SATURATE_FACTOR, sev)
    gray = (x * np.array([0.299, 0.587, 0.114], np.float32)).sum(-1, keepdims=True)
    return gray + (x - gray) * factor


def translate(x, sev, rng):
    frac = _tab(TRANSLATE_FRAC, sev)
    out = np.zeros_like(x)
    for i in range(len(x)):
        dy = int(rng.integers(-32, 33) * frac)
        dx = int(rng.integers(-32, 33) * frac)
        y0, y1 = max(0, dy), min(H, H + dy)
        x0, x1 = max(0, dx), min(W, W + dx)
        out[i, y0:y1, x0:x1] = x[i, y0 - dy : y1 - dy, x0 - dx : x1 - dx]
    return out


def rotate(x, sev, rng):
    deg = _tab(ROTATE_DEG, sev)
    out = np.empty_like(x)
    for i in range(len(x)):
        angle = rng.uniform(-deg, deg)
        out[i] = ndimage.rotate(x[i], angle, reshape=False, order=1, mode="constant", cval=0.0)
    return out


CORRUPTIONS = {
    "gaussian_noise": gaussian_noise,
    "shot_noise": shot_noise,
    "impulse_noise": impulse_noise,
    "defocus_blur": defocus_blur,
    "glass_blur": glass_blur,
    "motion_blur": motion_blur,
    "zoom_blur": zoom_blur,
    "snow": snow,
    "frost": frost,
    "fog": fog,
    "brightness": brightness,
    "contrast": contrast,
    "elastic": elastic,
    "pixelate": pixelate,
    "jpeg": jpeg_compression,
    "speckle_noise": speckle_noise,
    "gaussian_blur": gaussian_blur,
    "spatter": spatter,
    "saturate": saturate,
    "translate": translate,
    "rotate": rotate,
}

CORRUPTION_NAMES = list(CORRUPTIONS)


def corrupt_batch(images_uint8, name, sev, rng):
    """Apply one corruption to a uint8 batch; returns uint8."""
    x = images_uint8.astype(np.float32) / 255.0
    out = CORRUPTIONS[name](x, sev, rng)
    return np.clip(out * 255.0, 0, 255).astype(np.uint8)


def make_all(images_uint8, sev, rng):
    """Pick a random corruption type per image at a fixed severity."""
    out = np.empty_like(images_uint8)
    choice = rng.integers(0, len(CORRUPTION_NAMES), len(images_uint8))
    for k in np.unique(choice):
        m = choice == k
        out[m] = corrupt_batch(images_uint8[m], CORRUPTION_NAMES[k], sev, rng)
    return out

## 5. Training recipe

The two-layer filter banks are learned from the training images, the feature block
is stateless and computed once per augmented pool, a single `StandardScaler` +
`PCA` is fitted on that pool and shared by every member, and each member is also
added a second time behind a horizontal flip.

In [5]:
RECIPE = {
    "n_clean": 50000,        # all original training images
    "n_shift": 30000,        # flipped / translated copies
    "n_shift2": 20000,       # second, smaller flipped / translated copy set
    "n_corr13": 30000,       # corrupted copies, severity 1-3
    "n_corr15": 30000,       # corrupted copies, severity 1-5
    "n_corr_layered": 45000, # copies carrying two corruptions at once
    "learned": dict(n_l1=8, n_l2=8, patch=5, n_patch=120000, n_images=8000),
    "pca": 768,
    "hidden": (1024,),
    "epochs": 55,
    "alpha": 1e-4,
    "n_members": 3,          # bagged MLPs behind the shared scaler/PCA
    "prep_fit_rows": 60000,  # the scaler/PCA are fitted on this many rows
    "chunk": 20000,          # feature-extraction chunk size, caps peak memory
    "seeds": [42, 2026, 3102],
}

MODEL_FILES = {seed: f"model_seed{seed}.joblib" for seed in RECIPE["seeds"]}


def shift_view(dy, dx, flip, pad=4):
    """Reproduce one test-time view: optional horizontal flip, then a padded shift.

    Built from module-level NumPy callables only (pad + take), so the transform
    pickles by reference like everything else in the model.
    """
    steps = []
    if flip:
        steps.append(("hflip", _ft(np.flip, axis=2)))
    steps += [
        ("pad", _ft(np.pad, pad_width=((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="edge")),
        ("take_y", _ft(np.take, indices=np.arange(pad + dy, pad + dy + 32), axis=1)),
        ("take_x", _ft(np.take, indices=np.arange(pad + dx, pad + dx + 32), axis=2)),
    ]
    return Pipeline(steps)


def random_flip_shift(images, rng, max_shift=3):
    """Horizontal flip plus a small translation (edge-clamped)."""
    x = images.copy()
    flip = rng.random(len(x)) < 0.5
    if flip.any():
        x[flip] = x[flip][:, :, ::-1]
    dy = rng.integers(-max_shift, max_shift + 1, len(x))
    dx = rng.integers(-max_shift, max_shift + 1, len(x))
    rows = np.arange(len(x))[:, None, None]
    ys = np.clip(np.arange(32)[None, :, None] + dy[:, None, None], 0, 31)
    xs = np.clip(np.arange(32)[None, None, :] + dx[:, None, None], 0, 31)
    return x[rows, ys, xs]


def make_corrupted(images, rng, sev_lo, sev_hi):
    """Give every image a random corruption type and a random severity."""
    out = np.empty_like(images)
    choice = rng.integers(0, len(CORRUPTION_NAMES), len(images))
    sev = rng.integers(sev_lo, sev_hi + 1, len(images))
    for k in np.unique(choice):
        for s in np.unique(sev[choice == k]):
            m = (choice == k) & (sev == s)
            out[m] = corrupt_batch(images[m], CORRUPTION_NAMES[k], int(s), rng)
    return out


def transform_in_chunks(feats, images, chunk):
    """The feature block is stateless; running it in chunks keeps peak memory low."""
    parts = []
    for start in range(0, len(images), chunk):
        parts.append(feats.transform(images[start:start + chunk]).astype(np.float32))
    return np.concatenate(parts)


def build_model(seed, X_pool, y_pool, feats, recipe):
    """One submitted model: bagged soft-voting ensemble plus horizontal-flip TTA."""
    t0 = time.time()
    rng = np.random.default_rng(seed)

    idx_shift = rng.choice(len(X_pool), recipe["n_shift"], replace=False)
    idx_shift2 = rng.choice(len(X_pool), recipe["n_shift2"], replace=False)
    idx_corr13 = rng.choice(len(X_pool), recipe["n_corr13"], replace=False)
    idx_corr15 = rng.choice(len(X_pool), recipe["n_corr15"], replace=False)
    idx_layered = rng.choice(len(X_pool), recipe["n_corr_layered"], replace=False)

    names = ("clean", "shifted", "shifted2", "corrupted13", "corrupted15", "corrupted_twice")
    images = {
        "clean": X_pool,
        "shifted": random_flip_shift(X_pool[idx_shift], rng),
        "shifted2": random_flip_shift(X_pool[idx_shift2], rng),
        "corrupted13": make_corrupted(X_pool[idx_corr13], rng, 1, 3),
        "corrupted15": make_corrupted(X_pool[idx_corr15], rng, 1, 5),
        "corrupted_twice": make_corrupted(
            make_corrupted(X_pool[idx_layered], rng, 1, 3), rng, 1, 3
        ),
    }
    labels = {
        "clean": y_pool,
        "shifted": y_pool[idx_shift],
        "shifted2": y_pool[idx_shift2],
        "corrupted13": y_pool[idx_corr13],
        "corrupted15": y_pool[idx_corr15],
        "corrupted_twice": y_pool[idx_layered],
    }

    # every member sees the same augmented pool, so the features are extracted once
    blocks = []
    for name in names:
        block = transform_in_chunks(feats, images[name], recipe["chunk"])
        blocks.append(block)
        print(f"    features {name:12s} {block.shape}  {time.time() - t0:.0f}s", flush=True)
        images[name] = None
    X_all = np.concatenate(blocks)
    y_all = np.concatenate([labels[n] for n in names])
    del blocks, images, labels

    # one shared projection for every member: it is stored once in the .joblib file
    rows = np.random.default_rng(seed).choice(
        len(X_all), min(recipe["prep_fit_rows"], len(X_all)), replace=False
    )
    prep = Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=recipe["pca"], random_state=seed)),
    ]).fit(X_all[rows])
    del rows
    X_proj = prep.transform(X_all)
    X_all = None
    print(
        f"    {X_proj.shape[0]} samples x {X_proj.shape[1]} principal components"
        f"  {time.time() - t0:.0f}s",
        flush=True,
    )

    members = []
    for i in range(recipe["n_members"]):
        tm = time.time()
        head = MLPClassifier(
            hidden_layer_sizes=recipe["hidden"],
            alpha=recipe["alpha"],
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=recipe["epochs"],
            random_state=seed + i,
        ).fit(X_proj, y_all)
        inner = Pipeline([("prep", prep), ("head", head)])
        members.append(Pipeline([("features", feats), ("inner", inner)]))
        print(f"    member {i}: {head.n_iter_} epochs, {time.time() - tm:.0f}s", flush=True)

    # the flipped twins share the *fitted* head object, so they add no model size
    # Every member is evaluated on a grid of shifts, with and without a horizontal
    # flip. The view transform is a composition of module-level NumPy callables
    # (pad + take), and every view shares the *same fitted* head object, so the
    # whole view bank costs no model size - it is test-time augmentation only.
    views = [(0, 0, False)]
    for d in (1, 2, 3, 4):
        for sign in (-1, 1):
            for flip in (False, True):
                views += [(0, sign * d, flip), (sign * d, 0, flip)]

    print(f"    {len(views)} views x {len(members)} members "
          f"= {len(views) * len(members)} voting members", flush=True)

    view_members = []
    for dy, dx, flip in views:
        for i, m in enumerate(members):
            view_members.append(Pipeline([
                ("view", shift_view(dy, dx, flip)),
                ("features", feats),
                ("inner", m.named_steps["inner"]),
            ]))

    ensemble = VotingClassifier(
        [(f"m{i}", m) for i, m in enumerate(view_members)], voting="soft"
    )
    ensemble.estimators_ = view_members
    ensemble.le_ = LabelEncoder().fit(y_pool)
    ensemble.classes_ = ensemble.le_.classes_
    return ensemble


## 6. Train and save the three models

In [6]:
t0 = time.time()
filters = learn_two_layer_filters(X_train_images, seed=RECIPE["seeds"][0], **RECIPE["learned"])
print(f"learned {len(filters[0])} stage-1 filters and "
      f"{len(filters[1])} x {len(filters[1][0])} stage-2 filters  ({time.time() - t0:.0f}s)")

feats = build_feature_block(filters)
print("feature vector length:", feats.transform(X_train_images[:8]).shape[1])

summary = []
for seed in RECIPE["seeds"]:
    t0 = time.time()
    print(f"seed {seed}: building the training mixture")
    X_pool = X_train_images[: RECIPE["n_clean"]]
    y_pool = y_train[: RECIPE["n_clean"]]
    model = build_model(seed, X_pool, y_pool, feats, RECIPE)
    path = MODEL_FILES[seed]
    joblib.dump(model, path)
    size_mb = os.path.getsize(path) / 1e6
    summary.append((seed, size_mb, time.time() - t0))
    print(f"seed {seed}: saved {path}  |  {size_mb:.1f} MB  |  "
          f"{time.time() - t0:.0f}s", flush=True)


learned 8 stage-1 filters and 8 x 8 stage-2 filters  (2s)
feature vector length: 2496
seed 42: building the training mixture


    features clean        (50000, 2496)  117s


    features shifted      (30000, 2496)  173s


    features shifted2     (20000, 2496)  210s


    features corrupted13  (30000, 2496)  266s


...

D:\AIAA3102-migration\.venv314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (55) reached and the optimization hasn't converged yet.
  warnings.warn(


    member 2: 55 epochs, 856s


    33 views x 3 members = 99 voting members


seed 3102: saved model_seed3102.joblib  |  46.2 MB  |  3029s


D:\AIAA3102-migration\.venv314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (55) reached and the optimization hasn't converged yet.
  warnings.warn(


## 7. Verification: reload the saved models

In [7]:
# The saved models are test-time-augmented over a grid of shifts and flips, so this
# cell runs many forward passes; it is the slow part of the notebook and is also what
# Model_Evaluation.ipynb will do.
accuracies = []
for seed, path in MODEL_FILES.items():
    model = joblib.load(path)
    acc = accuracy_score(y_val, model.predict(X_val_images))
    accuracies.append(acc)
    print(f"{path}: validation accuracy {acc:.4f}  ({os.path.getsize(path) / 1e6:.1f} MB)")

print("")
print("Individual accuracies:")
for (seed, size_mb, secs), acc in zip(summary, accuracies):
    print(f"  model_seed{seed}.joblib: {acc:.4f}  ({size_mb:.1f} MB, {secs:.0f}s)")
print(f"Average validation accuracy: {np.mean(accuracies):.4f}")
print(f"All files below 100 MB: {all(os.path.getsize(p) / 1e6 < 100 for p in MODEL_FILES.values())}")


model_seed42.joblib: validation accuracy 0.6642  (46.2 MB)


model_seed2026.joblib: validation accuracy 0.6678  (46.2 MB)


model_seed3102.joblib: validation accuracy 0.6684  (46.2 MB)

Individual accuracies:
  model_seed42.joblib: 0.6642  (46.2 MB, 3060s)
  model_seed2026.joblib: 0.6678  (46.2 MB, 2990s)
  model_seed3102.joblib: 0.6684  (46.2 MB, 3029s)
Average validation accuracy: 0.6668
All files below 100 MB: True
